# Dropout: A Simple Way to Prevent Neural Networks from Overfitting (Srivastava et al., JMLR 2014)
### Triển Khai Từ Con Số 0 (From Scratch) Bằng Python & NumPy

> **Tác giả:** Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, Ruslan Salakhutdinov  
> **Tạp chí:** Journal of Machine Learning Research (JMLR) 15 (2014) 1929–1958  
> **Ý nghĩa lịch sử:** Vũ khí điều hòa kinh điển phá vỡ hiện tượng đồng thích ứng (co-adaptation) giữa các nơ-ron, tương đương việc huấn luyện trung bình tập hợp cấp số mũ $2^N$ mô hình con chia sẻ trọng số, góp phần quan trọng vào sự thành công của AlexNet và kỷ nguyên Deep Learning.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Cơ chế Dropout:** Tự lập trình cả hai biến thể: *Standard Dropout* (nhân hệ số $1 - p$ ở test time) và *Inverted Dropout* (chia trước cho $1 - p$ ở train time).
2. **Đo lường Co-adaptation:** Tính toán ma trận tương quan giữa các nơ-ron ẩn $\text{Corr}(h_i, h_j)$ để chứng minh Dropout ép nơ-ron học độc lập.
3. **Lan truyền ngược giải tích:** Đạo hàm qua tầng Inverted Dropout với mặt nạ nhị phân Bernoulli $\frac{\partial L}{\partial x} = \frac{\frac{\partial L}{\partial y} \odot \text{mask}}{1 - p}$.
4. **Academic Value Benchmark:** Đối chiếu định lượng 3 trường phái mô hình: Không điều hòa vs $L_2$ Weight Decay vs Dropout.

In [ ]:
import time
import numpy as np
print(f"[✓] Khởi tạo thư viện NumPy: {np.__version__}")

## PHẦN 1: CÁC KHỐI TẦNG NƠ-RON CƠ BẢN

Cài đặt tầng `LinearLayer` (có hỗ trợ $L_2$ Weight Decay), hàm kích hoạt `ReLULayer` và mất mát `SoftmaxCrossEntropy`.

In [ ]:
class LinearLayer:
    def __init__(self, in_features, out_features, l2_reg=0.0):
        self.in_features = in_features
        self.out_features = out_features
        self.l2_reg = l2_reg
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features).astype(np.float32) * scale
        self.b = np.zeros((1, out_features), dtype=np.float32)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.x = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, grad_output):
        self.dW = np.dot(self.x.T, grad_output) + self.l2_reg * self.W
        self.db = np.sum(grad_output, axis=0, keepdims=True)
        return np.dot(grad_output, self.W.T)

    def update_params(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db


class ReLULayer:
    def __init__(self):
        self.x = None

    def forward(self, x):
        self.x = x
        return np.maximum(0.0, x)

    def backward(self, grad_output):
        return grad_output * (self.x > 0).astype(np.float32)


class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.targets = None

    def forward(self, logits, targets):
        self.targets = targets
        N = logits.shape[0]
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        self.probs = exp_scores / (np.sum(exp_scores, axis=1, keepdims=True) + 1e-12)
        correct_logprobs = -np.log(self.probs[np.arange(N), targets] + 1e-12)
        return np.mean(correct_logprobs)

    def backward(self):
        N = self.probs.shape[0]
        grad = self.probs.copy()
        grad[np.arange(N), self.targets] -= 1.0
        grad /= N
        return grad

print("[✓] Đã khởi tạo các tầng Linear, ReLU, và SoftmaxCrossEntropy.")

## PHẦN 2: CÀI ĐẶT STANDARD DROPOUT vs INVERTED DROPOUT

- **Standard Dropout (Bài báo gốc 2014):**
  - Train: $\widetilde{y} = x \odot \text{mask}$, mask $\sim \text{Bernoulli}(1 - p)$.
  - Test: $y = x \cdot (1 - p)$ (Phải thay đổi trọng số/kích hoạt ở pha kiểm thử).

- **Inverted Dropout (Chuẩn mực hiện đại PyTorch/TensorFlow):**
  - Train: $\widetilde{y} = \frac{x \odot \text{mask}}{1 - p}$.
  - Test: $y = x$ (Giữ nguyên $100\%$ kiến trúc, zero-overhead khi suy luận thực tế!).

In [ ]:
class StandardDropoutLayer:
    def __init__(self, p=0.5):
        self.p = p
        self.p_keep = 1.0 - p
        self.mask = None
        self.mode = 'train'

    def forward(self, x):
        if self.mode == 'train' and self.p > 0:
            self.mask = (np.random.rand(*x.shape) < self.p_keep).astype(np.float32)
            return x * self.mask
        elif self.mode == 'eval' and self.p > 0:
            return x * self.p_keep
        return x

    def backward(self, grad_output):
        if self.mode == 'train' and self.p > 0:
            return grad_output * self.mask
        return grad_output


class InvertedDropoutLayer:
    def __init__(self, p=0.5):
        self.p = p
        self.p_keep = 1.0 - p
        self.mask = None
        self.mode = 'train'

    def forward(self, x):
        if self.mode == 'train' and self.p > 0:
            self.mask = (np.random.rand(*x.shape) < self.p_keep).astype(np.float32)
            return (x * self.mask) / self.p_keep
        return x

    def backward(self, grad_output):
        if self.mode == 'train' and self.p > 0:
            return (grad_output * self.mask) / self.p_keep
        return grad_output

print("[✓] Đã định nghĩa StandardDropoutLayer và InvertedDropoutLayer.")

## PHẦN 3: ĐO LƯỜNG ĐỊNH LƯỢNG ĐỒNG THÍCH ỨNG (CO-ADAPTATION)

Đo giá trị tuyệt đối trung bình của hệ số tương quan giữa các nơ-ron ẩn:
$$\text{Mean } |\text{Corr}| = \frac{1}{D(D - 1)} \sum_{i \neq j} |\text{Corr}(h_i, h_j)|$$

- **Không điều hòa:** Nơ-ron phụ thuộc lẫn nhau $\to$ Tương quan cao ($> 0.5$).
- **Có Dropout:** Nơ-ron học đặc trưng độc lập $\to$ Tương quan phân rã về gần $0$ ($< 0.2$).

In [ ]:
def compute_mean_coadaptation_correlation(hidden_activations):
    N, D = hidden_activations.shape
    centered = hidden_activations - np.mean(hidden_activations, axis=0, keepdims=True)
    std = np.std(hidden_activations, axis=0, keepdims=True) + 1e-8
    normalized = centered / std

    corr_matrix = np.dot(normalized.T, normalized) / N
    mask = ~np.eye(D, dtype=bool)
    off_diag_corrs = np.abs(corr_matrix[mask])
    return np.mean(off_diag_corrs)

print("[✓] Đã sẵn sàng hàm compute_mean_coadaptation_correlation.")

## PHẦN 4: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Huấn luyện trên bài toán dễ bị quá khớp (Dữ liệu ít $N_{\text{train}}=120$, số chiều lớn $D=40$ với 38 chiều nhiễu):
1. **Mô hình 1: Không Điều Hòa (No Regularization)**
2. **Mô hình 2: Điều hòa truyền thống ($L_2$ Weight Decay)**
3. **Mô hình 3: Dropout ($p = 0.5$) (Srivastava et al., 2014)**

In [ ]:
from dropout_scratch import run_academic_value_comparison

# Chạy kiểm thử đối chuẩn học thuật trực tiếp
run_academic_value_comparison()

## 🎓 TỔNG KẾT & BÀI HỌC LỊCH SỬ TỪ DROPOUT (JMLR 2014)

1. **Giải pháp đơn giản mà hiệu quả phi thường:** Chỉ với một phép nhân mặt nạ Bernoulli ngẫu nhiên, Dropout đã giải quyết được vấn nạn quá khớp nan giải của mạng nơ-ron sâu.
2. **Phá vỡ Co-adaptation:** Ép mỗi nơ-ron phải học cách tự trích xuất đặc trưng độc lập và mạnh mẽ, không thể ỷ lại vào nơ-ron bên cạnh.
3. **Bản chất Ensemble cấp số mũ:** Tương đương với việc huấn luyện trung bình $2^N$ mô hình con chia sẻ chung trọng số, mang lại sức mạnh của một hội đồng mô hình khổng lồ.
4. **Inverted Dropout:** Đưa việc chuẩn hóa về pha huấn luyện, giúp triển khai mô hình vào thực tế với chi phí tính toán suy luận hoàn toàn bằng 0 ($O(1)$ overhead).